In [ ]:
# for loop in EP_flux_Dycore_from_kai.ipynb
import h5py
import numpy as np
import sys
import importlib
import os

# === Add custom module path ===
sys.path.append("/data92/PeterChang/back_to_master1220/Moist_Dycore/Jucker_Climate/aostools/")
import climate
importlib.reload(climate)

# === Constants ===
last_n = 78000
PR_list = [0, 10, 20, 30, 40, 50]

# === Loop over different PR values ===
for PR in PR_list:
    print(f"Processing PR{PR}...")

    # === File paths ===
    base_dir = f"/data92/PeterChang/Dycore_data/PR{PR}"
    idx_dir = "/data92/PeterChang/back_to_master1220/Moist_Dycore/EPflux/New_PC1_time_idx"
    save_dir = "/data92/PeterChang/back_to_master1220/Moist_Dycore/EPflux/result"
    os.makedirs(save_dir, exist_ok=True)

    u_file = f"{base_dir}/u/PR{PR}_500_20000day_u_6hourly.h5"
    v_file = f"{base_dir}/v/PR{PR}_500_20000day_v_6hourly.h5"
    t_file = f"{base_dir}/t/PR{PR}_500_20000day_t_6hourly.h5"
    w_file = f"{base_dir}/w/PR{PR}_500_20000day_w_6hourly.h5"
    p_file = f"{base_dir}/p/PR{PR}_500_20000day_p_6hourly.h5"
    ps_file = f"{base_dir}/ps/PR{PR}_500_20000day_ps_6hourly.h5"

    # === Function to read last time steps ===
    def read_last_timesteps(file_path, last_n):
        with h5py.File(file_path, 'r') as f:
            key = list(f.keys())[0]
            return f[key][-last_n:, ...]

    # === Load variables ===
    u_last = read_last_timesteps(u_file, last_n)
    v_last = read_last_timesteps(v_file, last_n)
    t_last = read_last_timesteps(t_file, last_n)
    p_last = read_last_timesteps(p_file, last_n)

    # === Read PC1 indices ===
    with h5py.File(f"{idx_dir}/PC1_positive_1std_PR{PR}.h5", 'r') as f:
        pc1_pos_idx = f['time_idx_pos'][0]

    with h5py.File(f"{idx_dir}/PC1_negative_1std_PR{PR}.h5", 'r') as f:
        pc1_neg_idx = f['time_idx_neg'][0]

    # === Latitude and Pressure Setup ===
    y = np.linspace(-90, 90, u_last.shape[2])  # assume u(time, p, lat, lon)
    p1d_pos = p_last[pc1_pos_idx].mean(axis=(0, 2, 3))
    p1d_neg = p_last[pc1_neg_idx].mean(axis=(0, 2, 3))

    # === Compute EP flux for PC1 POSITIVE ===
    ep1_pos, ep2_pos, div1_pos, dthdp_pos, vptp_pos = climate.ComputeEPfluxDiv(
        y, p1d_pos / 100, u_last[pc1_pos_idx], v_last[pc1_pos_idx], t_last[pc1_pos_idx]
    )

    np.savez(f"{save_dir}/ep_flux_PR{PR}_positive.npz",
             ep1=ep1_pos, ep2=ep2_pos, y=y, p1d=p1d_pos)

    # === Compute EP flux for PC1 NEGATIVE ===
    ep1_neg, ep2_neg, div1_neg, dthdp_neg, vptp_neg = climate.ComputeEPfluxDiv(
        y, p1d_neg / 100, u_last[pc1_neg_idx], v_last[pc1_neg_idx], t_last[pc1_neg_idx]
    )

    np.savez(f"{save_dir}/ep_flux_PR{PR}_negative.npz",
             ep1=ep1_neg, ep2=ep2_neg, y=y, p1d=p1d_neg)

    print(f"Saved results for PR{PR}")



